# IAM: pair and token distributions

A simple comparison of our **handwriting-aware Bigram, BPE, and Unigram** tokenizers on the **5,694 IAM training lines** used to build them.

We look at three different things:
1. **Raw pairs:** which adjacent letters occur in the text, before tokenization?
2. **Vocabulary:** which pieces are available in each tokenizer?
3. **Actual usage:** which tokens does each encoder emit, and how often?

We also count supported, disconnected, and abstained image evidence. BPE and Unigram can contain longer pieces, such as `the` or `better`, so their distributions include more than pairs.

Run all cells with the `dtlr-poc` environment (pandas, matplotlib, IPython). This notebook reads the pinned IAM **train** exports and frozen models. It does not retrain a tokenizer, open raw images, or run DTLR/CCL again. These are descriptive training measurements, not recognition accuracy.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import gc
import hashlib
import json
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Use this checkout when launched from the repository or notebook directory.
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'poc/frozen/iam-english-handwriting-bpe-v1.json').is_file()), None)
if REPO is None:
    REPO = Path('//wsl.localhost/Ubuntu/home/artellisys/DTLR' if os.name == 'nt'
                else '/home/artellisys/DTLR')
IAM_TRAIN = Path('//wsl.localhost/Ubuntu/home/artellisys/dtlr-output/iam-train-full'
                 if os.name == 'nt' else '/home/artellisys/dtlr-output/iam-train-full')
OUTPUT = REPO / 'outputs/iam-tokenizer-distributions-v1'
assert REPO.is_dir() and OUTPUT.resolve().is_relative_to(REPO.resolve())
sys.path.insert(0, str(REPO / 'poc'))
from dtlr_poc.handwriting_subwords import (
    HandwritingSubwordTokenizer, prepare_corpus, span_connected,
    collect_candidates, extend_bigram_training_alphabet,
)
from dtlr_poc.tokenizer import HandwritingBigramTokenizer
from dtlr_poc.subword_freeze import SOURCE_SHA256, load_frozen_subword_model

COLORS = {'HW Bigram': '#2876a8', 'HW BPE': '#d67b27', 'HW Unigram': '#41936b'}
ORDER = list(COLORS)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False})
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_rows', 30)
pd.set_option('display.width', 150)

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


## 1. Load and verify the IAM training data

The image evidence was created earlier: **IAM line image + transcription → DTLR character boxes → character alignment → CCL → pair-facing component IDs**. Longer spans require one consistent component across their usable adjacent pairs.

We verify the input hashes and IAM/train identities before counting. We retain training text and pair evidence in memory; the export cell saves aggregate tables only.


In [ ]:
paths = {
    'selection': IAM_TRAIN / 'selection.json',
    'detections': IAM_TRAIN / 'detections.jsonl',
    'manifest': IAM_TRAIN / 'bigrams-dominant-core-v3/manifest.json',
    'scores': IAM_TRAIN / 'bigrams-dominant-core-v3/bigram_scores.json',
    'evidence': IAM_TRAIN / 'bigrams-dominant-core-v3/bigram_evidence.json',
    'frozen_bigram': REPO / 'poc/frozen/iam-english-handwriting-bigram-v1.json',
}
source_hashes = {name: sha256(path) for name, path in paths.items()}
assert source_hashes == SOURCE_SHA256, 'A pinned input changed.'
selection, manifest = read_json(paths['selection']), read_json(paths['manifest'])
records = [json.loads(line) for line in paths['detections'].read_text(encoding='utf-8').splitlines() if line.strip()]
evidence = read_json(paths['evidence'])
corpus = prepare_corpus(records, evidence, selection, manifest)
scores = read_json(paths['scores'])
assert len(scores) == manifest['score_count']
del records, evidence
gc.collect()
characters = Counter(c for line in corpus for c in line['text'])
character_count = sum(characters.values())
assert len(corpus) == 5694 and character_count == 241876 and len(characters) == 79
display(pd.DataFrame({'Measure': ['IAM training lines', 'Characters including spaces', 'Unique characters'],
                      'Count': [len(corpus), character_count, len(characters)]}))


## 2. Load the three handwriting-aware tokenizers

The frozen Bigram has 76 singleton characters. The full training alphabet has 79: it also needs `#`, `&`, and `(`. We use the existing **derived alphabet adapter in memory**, preserving all 145 bigrams, original IDs, scores, and DP segmentation. The frozen file is unchanged.

All three comparison tokenizers therefore have **79 singleton characters + 145 multi-character pieces + CTC blank = 225 classes**. Blank is excluded from every distribution because encoding must never emit it.

Bigram uses utility-maximizing DP; BPE applies its learned merges in order; Unigram chooses the most probable token sequence with Viterbi. This notebook compares their declared encoders.


In [ ]:
frozen_bigram = read_json(paths['frozen_bigram'])
bigram_model = extend_bigram_training_alphabet(frozen_bigram, set(characters))
bpe_model = load_frozen_subword_model(REPO / 'poc/frozen/iam-english-handwriting-bpe-v1.json')
unigram_model = load_frozen_subword_model(REPO / 'poc/frozen/iam-english-handwriting-unigram-v1.json')
models = dict(zip(ORDER, [bigram_model, bpe_model, unigram_model]))
tokenizers = {
    'HW Bigram': HandwritingBigramTokenizer(bigram_model),
    'HW BPE': HandwritingSubwordTokenizer(bpe_model),
    'HW Unigram': HandwritingSubwordTokenizer(unigram_model),
}
model_paths = [paths['frozen_bigram'], REPO / 'poc/frozen/iam-english-handwriting-bpe-v1.json',
               REPO / 'poc/frozen/iam-english-handwriting-unigram-v1.json']
model_hashes = {path.name: sha256(path) for path in model_paths}
pieces = {name: {t for t in model['vocab'] if len(t) > 1} for name, model in models.items()}
assert bigram_model['vocabulary'] == frozen_bigram['vocabulary']
assert all(bigram_model['vocab'][t] == i for t, i in frozen_bigram['vocab'].items())
assert all(len(tokens) == 145 for tokens in pieces.values())
assert all(tok.size == 225 for tok in tokenizers.values())
model_table = pd.DataFrame([
    {'Tokenizer': name, 'Singletons': sum(len(t) == 1 for t in model['vocab']),
     'Multi-character pieces': len(pieces[name]), 'CTC classes': model['size'],
     'Longest piece': max(map(len, pieces[name]))}
    for name, model in models.items()
])
display(model_table)
print('Bigram alphabet additions:', bigram_model['alphabet_adaptation']['added_singletons'])


## 3. Raw adjacent-letter pair distribution

Here `the` contributes both `th` and `he`, even if an encoder later emits `the` as one token. Counts are **case-sensitive** and include overlapping occurrences. We count pairs within each line; pairs containing spaces, punctuation, or digits are excluded from this letter-pair view.

**Supported:** exact usable character alignment and both pair-facing cores share a component. **Disconnected:** exact usable alignment, but different components. **Abstained:** missing, non-exact, or unusable evidence. Abstained occurrences stay outside the connectivity-rate denominator.


In [ ]:
pair_counts = defaultdict(Counter)
for line in corpus:
    for start, edge in line['edges'].items():
        pair = line['text'][start:start + 2]
        state = 'abstained' if edge is None else ('supported' if edge[0] == edge[1] else 'disconnected')
        pair_counts[pair][state] += 1
# Reconstruct the pinned aggregate scores from occurrence-level evidence.
assert set(pair_counts) == {row['pair'] for row in scores}
for row in scores:
    count = pair_counts[row['pair']]
    exact = count['supported'] + count['disconnected']
    assert sum(count.values()) == row['n_total']
    assert exact == row['n_exact_alignment']
    assert (count['supported'] / exact if exact else None) == row['exact_alignment_connected_rate']

pair_table = pd.DataFrame([
    {'pair': pair, 'raw_occurrences': sum(count.values()),
     'supported': count['supported'], 'disconnected': count['disconnected'],
     'abstained': count['abstained'],
     'usable_occurrences': count['supported'] + count['disconnected'],
     'connected_rate': count['supported'] / (count['supported'] + count['disconnected'])
                       if count['supported'] + count['disconnected'] else np.nan,
     **{name: pair in pieces[name] for name in ORDER}}
    for pair, count in sorted(pair_counts.items()) if all(c.isalpha() for c in pair)
]).sort_values(['raw_occurrences', 'pair'], ascending=[False, True]).reset_index(drop=True)
assert (pair_table['supported'] + pair_table['disconnected'] + pair_table['abstained']).equals(pair_table['raw_occurrences'])
assert set(pair_table.loc[(pair_table.usable_occurrences >= 20) & (pair_table.connected_rate >= 0.5), 'pair']) == pieces['HW Bigram']
print(f'{len(pair_table):,} distinct letter pairs; {pair_table.raw_occurrences.sum():,} raw occurrences.')
display(pair_table.head(15).round({'connected_rate': 3}))

fig, ax = plt.subplots(figsize=(9, 5))
top = pair_table.head(20).iloc[::-1]
ax.barh(top['pair'], top['raw_occurrences'],
        color=[COLORS['HW Bigram'] if chosen else '#b9c2ca' for chosen in top['HW Bigram']])
ax.set(title='20 most frequent adjacent letter pairs in IAM train', xlabel='Raw occurrences (before tokenization)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=COLORS['HW Bigram'], label='Selected HW bigram'),
                   Patch(color='#b9c2ca', label='Outside HW bigram vocabulary')], loc='lower right')
fig.tight_layout()
plt.show()


## 4. Frequency and handwriting support are different

Each point below is a distinct letter pair. The horizontal axis counts usable exact-alignment observations; the vertical axis is the fraction connected. A pair enters the frozen Bigram vocabulary only when **count ≥ 20 and connected rate ≥ 0.5**.

A frequent pair need not be mostly connected. Gray points may have too few usable observations or too little support. These frozen thresholds are experimental settings, not statistically optimal cutoffs.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for selected, color, label in [(False, '#b9c2ca', 'Not selected'), (True, COLORS['HW Bigram'], 'Selected HW bigram')]:
    group = pair_table[(pair_table['HW Bigram'] == selected) & (pair_table.usable_occurrences > 0)]
    ax.scatter(group.usable_occurrences, group.connected_rate, s=23, alpha=0.65, color=color, label=label)
ax.axvline(20, color='#555555', linestyle='--', linewidth=1)
ax.axhline(0.5, color='#555555', linestyle='--', linewidth=1)
ax.set(xscale='log', ylim=(-0.03, 1.03), title='Which letter pairs meet the handwriting evidence policy?',
       xlabel='Usable exact-alignment observations (log scale)', ylabel='Connected fraction of usable observations')
ax.legend()
fig.tight_layout()
plt.show()
print('Pairs with no usable evidence (omitted from scatter):', int((pair_table.usable_occurrences == 0).sum()))


## 5. Vocabulary pieces by length

This counts **unique pieces in the vocabulary**, regardless of how frequently they are used. A vocabulary containing `the` has one three-character piece. It does not add a separate `th` or `he` vocabulary entry unless that pair was also selected.

The shared image-supported candidate pool contains 303 pieces of length 2–6. BPE chooses pieces through supported merges; Unigram starts from handwriting-supported candidates and fits/prunes a text likelihood model. Each keeps 145 pieces.


In [ ]:
candidates = collect_candidates(corpus, max_length=6, minimum_count=20, rate_threshold=0.5)
assert len(candidates) == 303
for name in ('HW BPE', 'HW Unigram'):
    assert all(row == candidates[row['token']] for row in models[name]['vocabulary'])
assert all(tokens <= set(candidates) for tokens in pieces.values())
vocabulary_lengths = pd.DataFrame({name: Counter(map(len, tokens)) for name, tokens in pieces.items()})
vocabulary_lengths = vocabulary_lengths.reindex(range(2, 7)).fillna(0).astype(int)
vocabulary_lengths.index.name = 'Characters per piece'
display(vocabulary_lengths)
fig, ax = plt.subplots(figsize=(8, 4))
vocabulary_lengths.plot.bar(ax=ax, color=[COLORS[name] for name in ORDER], rot=0)
ax.set(title='Selected multi-character vocabulary by length', ylabel='Distinct vocabulary pieces', xlabel='Characters per piece')
fig.tight_layout()
plt.show()


## 6. Encode all IAM training lines and count actual usage

Vocabulary membership does not guarantee that a piece is emitted. Overlapping choices compete: `the` may become `[t, he]` or `[the]`, depending on the encoder.

We count every emitted token, including singleton punctuation and spaces. We also check exact reconstruction and no blank emission on every line. For multi-character tokens, we reuse the saved component evidence at their actual offsets.

For longer pieces, **supported** requires one component across the entire span. **Disconnected** means usable evidence fails this same-component rule; that can include inconsistent component IDs across neighboring cores. **Abstained** means at least one required pair is unusable or non-exact. These are automatic evidence categories, not human ground-truth labels.


In [ ]:
usage, used_lines, span_status = {}, {}, {}
summary_rows = []
for name, tokenizer in tokenizers.items():
    counts, line_counts = Counter(), Counter()
    status_counts = defaultdict(Counter)
    for line in corpus:
        result = tokenizer.segment(line['text'])
        tokens = result['tokens']
        ids = [tokenizer.vocab[token] for token in tokens]
        assert tokenizer.decode(ids) == line['text'] and 0 not in ids
        counts.update(tokens)
        line_counts.update(set(tokens))
        for segment in result['segments']:
            token = segment['token']
            assert token == line['text'][segment['start']:segment['end']]
            if len(token) > 1:
                supported = span_connected(line, segment['start'], segment['end'])
                state = 'abstained' if supported is None else ('supported' if supported else 'disconnected')
                status_counts[token][state] += 1
    usage[name], used_lines[name], span_status[name] = counts, line_counts, status_counts
    assert sum(len(token) * count for token, count in counts.items()) == character_count
    assert '' not in counts
    assert all(sum(status_counts[token].values()) == counts[token] for token in pieces[name])
    summary_rows.append({'Tokenizer': name, 'Emitted tokens': sum(counts.values()),
                         'Reduction vs characters (%)': 100 * (1 - sum(counts.values()) / character_count),
                         'Distinct tokens used': len(counts),
                         'Multi-character occurrences': sum(n for t, n in counts.items() if len(t) > 1),
                         'Unused multi-character pieces': sum(counts[t] == 0 for t in pieces[name])})
summary_table = pd.DataFrame(summary_rows).set_index('Tokenizer')
display(summary_table.round(2))

usage_lengths = pd.DataFrame({name: {length: sum(n for t, n in usage[name].items() if len(t) == length)
                                    for length in range(1, 7)} for name in ORDER})
usage_lengths.index.name = 'Characters per emitted token'
usage_percent = usage_lengths.div(usage_lengths.sum(), axis=1) * 100
display(usage_lengths)
fig, ax = plt.subplots(figsize=(8, 4))
usage_percent.plot.bar(ax=ax, color=[COLORS[name] for name in ORDER], rot=0)
ax.set(title='How often do encoders emit each token length?', xlabel='Characters per emitted token',
       ylabel='Share of all emitted tokens (%)')
fig.tight_layout()
plt.show()


## 7. Most frequently emitted multi-character tokens

These plots show **actual segmented usage**, which differs from raw substring frequency. Spaces and singleton characters are excluded here so that the learned multi-character pieces are easy to compare. `<SPACE>` appears explicitly in the complete table below.

The table also gives the raw overlapping frequency of each piece before encoding. A piece's emitted count can be lower because another overlapping piece wins.


In [ ]:
token_rows = []
for name in ORDER:
    for token in sorted(models[name]['vocab']):
        if token == '':
            continue
        evidence_row = candidates.get(token)
        status = span_status[name].get(token, Counter())
        known = status['supported'] + status['disconnected']
        token_rows.append({
            'Tokenizer': name, 'token': token, 'label': '<SPACE>' if token == ' ' else token,
            'length': len(token), 'raw_occurrences': characters[token] if len(token) == 1 else evidence_row['n_total'],
            'emitted_occurrences': usage[name][token], 'lines_using_token': used_lines[name][token],
            'percent_of_emitted_tokens': 100 * usage[name][token] / sum(usage[name].values()),
            'supported': status['supported'] if len(token) > 1 else np.nan,
            'disconnected': status['disconnected'] if len(token) > 1 else np.nan,
            'abstained': status['abstained'] if len(token) > 1 else np.nan,
            'connected_rate_all_raw_usable': evidence_row['connected_rate'] if evidence_row else np.nan,
            'connected_rate_emitted_usable': status['supported'] / known if known else np.nan,
        })
token_table = pd.DataFrame(token_rows)
assert len(token_table) == 3 * 224
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharex=True)
for name, ax in zip(ORDER, axes):
    top = token_table[(token_table.Tokenizer == name) & (token_table.length > 1)].sort_values(
        ['emitted_occurrences', 'token'], ascending=[False, True]).head(15).iloc[::-1]
    ax.barh(top.label, top.emitted_occurrences, color=COLORS[name])
    ax.set(title=name, xlabel='Emitted occurrences')
fig.suptitle('15 most used multi-character tokens in IAM train')
fig.tight_layout()
plt.show()
display(token_table[['Tokenizer', 'label', 'length', 'raw_occurrences', 'emitted_occurrences',
                     'lines_using_token']].sort_values(['Tokenizer', 'emitted_occurrences'], ascending=[True, False])
        .groupby('Tokenizer', sort=False).head(8).reset_index(drop=True))


## 8. How much emitted-token evidence is supported or abstained?

The stacked bars use **all emitted multi-character occurrences** as the denominator, including abstentions. The table separately reports **supported / (supported + disconnected)**, so you can see the effect of excluding missing evidence.

A vocabulary can pass the 0.5 rate threshold yet produce disconnected or abstained occurrences: selection is an aggregate decision, and inference applies the learned model to text without checking each image.


In [ ]:
status_table = pd.DataFrame({name: {state: sum(count[state] for count in span_status[name].values())
                                   for state in ['supported', 'disconnected', 'abstained']} for name in ORDER}).T
assert status_table.sum(axis=1).equals(summary_table['Multi-character occurrences'])
status_report = status_table.copy()
status_report['supported_percent_of_usable'] = 100 * status_report.supported / (
    status_report.supported + status_report.disconnected)
status_report['abstained_percent_of_all'] = 100 * status_report.abstained / status_table.sum(axis=1)
display(status_report.round(2))
fig, ax = plt.subplots(figsize=(9, 3.5))
(status_table.div(status_table.sum(axis=1), axis=0) * 100).plot.barh(
    stacked=True, ax=ax, color=['#41936b', '#d67b27', '#b9c2ca'])
ax.set(title='Image evidence for emitted multi-character occurrences', xlabel='Share of all multi-character occurrences (%)',
       ylabel='', xlim=(0, 100))
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=3)
fig.tight_layout()
plt.show()


## 9. Which exact two-character pieces are shared?

BPE and Unigram may keep fewer two-character pieces because part of their budget goes to longer pieces. This table compares **exact pair entries**. A pair inside a longer piece is not counted as a separate vocabulary entry.


In [ ]:
pair_sets = {name: {t for t in pieces[name] if len(t) == 2} for name in ORDER}
pair_overlap = pd.DataFrame({right: {left: len(pair_sets[left] & pair_sets[right])
                                    for left in ORDER} for right in ORDER})
print('Diagonal = pair vocabulary size; other cells = shared exact pairs.')
display(pair_overlap)
shared_pairs = sorted(set.intersection(*pair_sets.values()))
print(f'Pairs present in all three vocabularies ({len(shared_pairs)}):')
print(', '.join(shared_pairs))


## 10. Small segmentation examples

These are illustrative words, not an evaluation sample. `|` marks token boundaries. You can edit the list and rerun this cell. Notice how the same word can contain different token units across the three models.


In [ ]:
examples = ['the', 'better', 'letter', 'handwriting', 'evidence']
example_table = pd.DataFrame([
    {'Text': text, **{name: ' | '.join(tokenizers[name].segment(text)['tokens']) for name in ORDER}}
    for text in examples
])
display(example_table)


## 11. Save the tables

CSV tables and a summary stay inside `DTLR/outputs/iam-tokenizer-distributions-v1/`. These contain aggregate counts, not the full training transcriptions. The full `token_distribution.csv` includes every nonblank vocabulary piece, even those used zero times.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
for filename, table, include_index in [
    ('letter_pair_distribution.csv', pair_table, False),
    ('token_distribution.csv', token_table, False),
    ('vocabulary_lengths.csv', vocabulary_lengths, True),
    ('emitted_token_lengths.csv', usage_lengths, True),
    ('tokenizer_summary.csv', summary_table, True),
    ('emitted_span_evidence.csv', status_report, True),
    ('pair_vocabulary_overlap.csv', pair_overlap, True),
]:
    table.to_csv(OUTPUT / filename, index=include_index)
analysis_summary = {
    'dataset': 'IAM', 'split': 'train', 'scope': 'descriptive training distributions',
    'line_count': len(corpus), 'character_count': character_count,
    'distinct_letter_pairs': len(pair_table), 'raw_letter_pair_occurrences': int(pair_table.raw_occurrences.sum()),
    'eligible_candidate_count': len(candidates),
    'bigram_comparison': 'in-memory IAM-train alphabet adaptation; frozen file unchanged',
    'source_sha256': source_hashes, 'frozen_model_sha256': model_hashes,
    'summary': summary_table.reset_index().to_dict(orient='records'),
    'connectivity_denominator': 'usable exact-alignment occurrences; abstentions reported separately',
    'recognition_accuracy_measured': False,
}
(OUTPUT / 'summary.json').write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert all(sha256(paths[key]) == value for key, value in source_hashes.items())
assert all(sha256(path) == model_hashes[path.name] for path in model_paths)
print('Saved 7 CSV tables and summary.json to', OUTPUT)
print('All 5,694 lines round-trip for each tokenizer; no blank emitted; all input/model hashes unchanged.')


## How to read the comparison

- **Pair frequency** describes the training text; **connected rate** describes the usable handwriting evidence.
- **Vocabulary length distribution** counts distinct available pieces; **usage length distribution** counts emitted occurrences. They answer different questions.
- Bigram's DP, BPE's merge order, and Unigram's probabilities affect how overlapping pieces are used. Unigram probabilities are not observed token frequencies.
- Abstained does not mean disconnected. Raw candidate support and support after segmentation also use different sets of occurrences.
- These models and statistics use the same IAM training evidence. They show distributions and reproducibility, not independent validation or better CER/WER. The longer-span evidence rule has localization limitations.
